In [1]:
# import anything we will need to use
# ALC
# Imports

import pandas as pd 
from os import walk
import re
import seaborn as sns
import matplotlib.pyplot as plt
import datetime
import math

In [2]:
#ALC
# Speed Restriction Data from MBTA Open Data Portal
# https://mbta-massdot.opendata.arcgis.com/datasets/d73ed67e4cc84a84b818ea2c5caef696/about
slow_zone_files = []
# walk through the directory for slow zone data and read it all in
for (root, dirs, filenames) in walk("../data/slow zones/"):
    slow_zone_files.extend(filenames)
    break
    
slow_zone_data = []

# traverse and read in slow zones data
for file in slow_zone_files:
    path = f"../data/slow zones/{file}"
    if (re.search("\.csv$", file)):
        sz = pd.read_csv(path, quotechar='"')
        slow_zone_data.append(sz)
        
slow_zones = pd.concat(slow_zone_data)
slow_zones.shape

(127922, 32)

In [3]:
# ALC
# these are as downloaded from the MBTA's Open Data Portal. There is one file
# for every month of the year. 
# https://mbta-massdot.opendata.arcgis.com/datasets/90ed9092bd7a4285b296d5ff938edf29_0/explore
# https://mbta-massdot.opendata.arcgis.com/datasets/f960d5089e164fb8b6c6936c70719d72/about
# https://mbta-massdot.opendata.arcgis.com/datasets/ef115f7cf6684360b040b6d1d033eff0/about
# initialize array
alerts = []
alerts_data = []

# we can get a list of all the filepaths we need to read in
for year in [2022, 2023, 2024]:
    for (root, dirs, filenames) in walk(f"../data/service alerts/{year}"):
        for filename in filenames:
            alerts.append(f"{root}/{filename}")
    
# this will give us back a list of dataframes
# as a result of reading in these files
for file in alerts:
    # if there is something like a .DS_STORE, we don't 
    # want to try to read that in. we know we have CSVs
    if (re.search("\.csv$", file)):
        ad = pd.read_csv(file, quotechar='"', low_memory = False)
        alerts_data.append(ad)
    
# we combine our list of dataframes
service_alerts = pd.concat(alerts_data)

# how many rows and columns do we have?
service_alerts.shape

(2095409, 27)

In [4]:
# ALC
# MBTA Reliability data from the MBTA Open Data Portal
# https://mbta-massdot.opendata.arcgis.com/datasets/b3a24561c2104422a78b593e92b566d5_0/explore
reliability_input = "../data/reliability/MBTA Bus, Commuter Rail, & Rapid Transit Reliability.csv"
reliability = pd.read_csv(reliability_input)

reliability.shape

(963838, 13)

In [5]:
# ALC
# Gated Station Entries
# From the MBTA Open Data Portal
# https://mbta-massdot.opendata.arcgis.com/datasets/001c177f07594e7c99f193dde32284c9_0/explore
gse_files = []
gse_data = []
# walk through the directory for slow zone data and read it all in
for (root, dirs, filenames) in walk("../data/gated station entries/"):
    for filename in filenames:
        gse_files.append(f"{root}/{filename}")
    
# traverse and read in slow zones data
for file in gse_files:
    if (re.search("\.csv$", file)):
        gse = pd.read_csv(file, quotechar='"')
        gse_data.append(gse)
        
station_entries = pd.concat(gse_data)
station_entries.shape

(3731053, 7)

In [6]:
# ALC
# custom functions we will use later
# this is a simple function to ensure that what
# the columns we are working with are datetime objects
def get_date_column(date):
    return pd.to_datetime(date, format='mixed').dt.date

# this ensures that the line column
# is consistent across datasets
def simplify_route(line):
    if pd.isna(line) or 'Silver Line' in line:
        return None
    elif 'Green Line' in line or 'Green-' in line:
        return 'green'
    elif 'Red Line' in line or 'Red' in line:
        return "red"
    elif 'Blue Line' in line or 'Blue' in line:
        return 'blue'
    elif 'Orange Line' in line or 'Orange' in line:
        return 'orange'
    else:
        return line
    
# function to aggregate reliability data by date and line
# get one row per line per date with an aggregate score
def agg_reliability_data_daily(reliability):
    return reliability.groupby(['service_date', 'line'], as_index=False)[
        ['otp_numerator', 'otp_denominator']
    ].sum()
    
# gated station entries are given by the half hour. we will need to
# aggregate these down to the day
def agg_station_entries_daily(station_entries):
    return station_entries.groupby(['service_date', 'line'], as_index=False)[
               'est_ridership'
           ].sum()

# count and distane of slow zones daily
def agg_slow_zones_daily(slow_zones):
    return slow_zones.groupby(['service_date', 'line'], as_index=False).agg(
        num_slow_zones = ('ID', 'nunique'),
        total_track_pct = ('Line_Restricted_Track_Pct', 'sum'),
        total_miles_affected = ('Restriction_Distance_Miles', 'sum')
    )

# move alert data to wide format with counts for cause/effect
def pivot_alerts(df):
    # so we need a count of different kinds of effects of alerts
    # i think this must be done one column at a time
    effect = pd.pivot_table(
        expanded_alerts_df,
        values = 'count',
        index = ['service_date', 'line'],
        columns = 'alert_effect',
        aggfunc = 'sum',
        # we'll assume no data = 0 alerts with that effect
        fill_value = 0
    )
    
    # rename columns to indicate effect
    effect.columns = [f"alert_effect_{col}" for col in effect.columns]
    
    # then we can do cause
    cause = pd.pivot_table(
        expanded_alerts_df,
        values = 'count',
        index = ['service_date', 'line'],
        columns = 'alert_cause',
        aggfunc = 'sum',
        # same as effect
        fill_value = 0
    )
    # rename columns to indicate cause
    cause.columns = [f"alert_cause_{col}" for col in cause.columns]
    
    # mix it all together
    cause_effect = pd.concat([cause, effect], axis = 1)
    
    # if we don't reset_index, it's indexed by date
    # but we just need our date/line combos
    return cause_effect.reset_index()

In [7]:
# ALC
# make date format consistent in all date columns
# the date in the reliability data appears yyyy/mm/dd hh:mm:ss+00
# we will use yyyy-mm-dd for service date
reliability['service_date'] = get_date_column(
    reliability['service_date'])

# slow zones uses what we expect but we just rename the column for consistency
slow_zones['Calendar_Date'] = get_date_column(
    slow_zones['Calendar_Date'])
slow_zones = slow_zones.rename(columns = {'Calendar_Date': 'service_date'})

service_alerts['active_period_start_dt'] = get_date_column(
    service_alerts['active_period_start_dt'])
service_alerts['active_period_end_dt'] = get_date_column(
    service_alerts['active_period_end_dt'])
station_entries['service_date'] = get_date_column(
    station_entries['service_date'])

In [8]:
# ALC
# station entry lines
station_entries['route_or_line'] = station_entries['route_or_line'].apply(simplify_route)
rt_lines = ['green', 'red', 'blue', 'orange']

# there is silver line data in here which is out of scope
station_entries = station_entries[station_entries['route_or_line'].isin(rt_lines)]
station_entries = station_entries.rename(columns = {'gated_entries': 'est_ridership',
                                                   'route_or_line': 'line'})

# slow zones lines
slow_zones['line'] = slow_zones['Line'].apply(simplify_route)

# reliability lines, cr and rt
reliability['line'] = reliability['gtfs_route_long_name'].apply(simplify_route)

# this removes out of scope bus data
reliability = reliability[reliability['line'].notna()]

service_alerts['route_id'] = service_alerts['route_id'].apply(simplify_route)
service_alerts = service_alerts.rename(columns = {'route_id': 'line'})
service_alerts['line'] = service_alerts['line'].astype(str).apply(simplify_route)
# there are a lot of bus rows here
service_alerts = service_alerts[service_alerts['line'].isin(rt_lines)]
service_alerts.shape

(373220, 27)

In [9]:
# JJ
# Expand service alerts and prep columns

service_alerts['active_period_start_dt'] = get_date_column(service_alerts['active_period_start_dt'])
service_alerts['active_period_end_dt'] = get_date_column(service_alerts['active_period_end_dt'])

# ALC refactored a bit and moved the restriction to rt lines up a few cells

In [10]:
# JJ
# Deduplicate alerts by alert_id first
service_alerts = service_alerts.drop_duplicates(subset=["alert_id"])

In [11]:
# JJ
# Expand alerts by date

expanded_alerts = []
for _, row in service_alerts.iterrows():
    if pd.isna(row['active_period_start_dt']) or pd.isna(row['active_period_end_dt']):
        continue
    for d in pd.date_range(row['active_period_start_dt'], row['active_period_end_dt']):
        expanded_alerts.append({
            'service_date': d.date(),
            'line': row['line'],
            # ALC added
            'alert_cause': row['cause_detail'],
            'alert_effect': row['effect_detail'],
            'count': 1
        })

expanded_alerts_df = pd.DataFrame(expanded_alerts)

In [12]:
# JJ
# Aggregate counts

daily_alert_counts = (
    expanded_alerts_df
    .groupby(['service_date', 'line'], as_index=False)
    .size()
    .rename(columns={'size': 'num_alerts'})
)

In [13]:
# ALC
alerts_cause_effect = pivot_alerts(expanded_alerts_df)
# some of these can probably be collapsed
alerts_cause_effect.head(10)

,service_date,line,alert_cause_ACCIDENT,alert_cause_CONSTRUCTION,alert_cause_DISABLED_TRAIN,alert_cause_FIRE,alert_cause_FIRE_DEPARTMENT_ACTIVITY,alert_cause_MAINTENANCE,alert_cause_MEDICAL_EMERGENCY,alert_cause_POLICE_ACTION,...,alert_effect_PARKING_CLOSURE,alert_effect_PARKING_ISSUE,alert_effect_SCHEDULE_CHANGE,alert_effect_SERVICE_CHANGE,alert_effect_SHUTTLE,alert_effect_STATION_CLOSURE,alert_effect_STATION_ISSUE,alert_effect_STOP_CLOSURE,alert_effect_STOP_MOVE,alert_effect_SUSPENSION
0,2022-01-01,green,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2022-01-01,orange,0,0,0,0,0,1,0,1,...,0,0,0,1,0,0,0,0,0,0
2,2022-01-01,red,0,0,0,0,0,0,2,0,...,0,0,0,0,1,0,0,0,0,0
3,2022-01-02,orange,0,0,0,0,0,1,0,1,...,0,0,0,1,0,0,0,0,0,0
4,2022-01-03,green,0,3,1,0,0,1,0,0,...,0,0,0,0,4,0,0,0,0,0
5,2022-01-03,orange,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,2022-01-04,green,0,3,0,0,0,1,0,0,...,0,0,0,0,4,0,0,0,0,0
7,2022-01-04,orange,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,2022-01-04,red,0,0,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,2022-01-05,blue,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [14]:
# ALC
# join datasets
# does it have to be done two at a time?
station_entries_reliability = pd.merge(
    agg_station_entries_daily(station_entries), 
    agg_reliability_data_daily(reliability), 
    on = ['service_date', 'line'], 
    # inner because both these sets don't really have
    # missing data, and missing data implies incompleteness here
    # we would have had to handle it specially
    how = 'inner')
    
# doing a left join here. if there is no data in the 
# slow zones file, it means slow zones = none
merged_data = pd.merge(
    station_entries_reliability,
    agg_slow_zones_daily(slow_zones),
    on = ['service_date', 'line'],
    how = 'left'
)

# we have complete data from 2023+
merged_data = merged_data[merged_data['service_date'] > datetime.date(2022, 12, 31)]
# assign day of week, month, and otp score columns
merged_data['day_of_week'] = pd.to_datetime(merged_data['service_date']).dt.day_name()
merged_data['month'] = pd.to_datetime(merged_data['service_date']).dt.month_name()
merged_data['otp_score'] = merged_data['otp_numerator'] / merged_data['otp_denominator']

In [15]:
# JJ
# Merge with merged data

final_data_v2 = pd.merge(
    merged_data,
    daily_alert_counts,
    on=["service_date", "line"],
    how="left"
)

final_data_v2.head(5)

,service_date,line,est_ridership,otp_numerator,otp_denominator,num_slow_zones,total_track_pct,total_miles_affected,day_of_week,month,otp_score,num_alerts
0,2023-01-01,blue,15214.400000,33536.796260,33747.30964,NaN,NaN,NaN,Sunday,January,0.993762,NaN
1,2023-01-01,green,22745.689929,46343.320825,58270.10887,17.0,0.032971,1.780114,Sunday,January,0.795319,NaN
2,2023-01-01,orange,24492.170000,45674.325060,46679.88702,12.0,0.088899,2.003788,Sunday,January,0.978458,2.0
3,2023-01-01,red,27863.440000,59984.367440,66606.05448,23.0,0.074641,3.538731,Sunday,January,0.900584,NaN
4,2023-01-02,blue,19025.400000,67706.968410,68626.25452,NaN,NaN,NaN,Monday,January,0.986604,NaN


In [16]:
# JJ
# Check service alert count per line
service_alerts['line'].value_counts(dropna=False)

line
red       2681
green     2178
orange    1337
blue      1283
Name: count, dtype: int64

In [17]:
# ALC
# lack of service alerts, fill with 0
final_data_v2['num_alerts'] = final_data_v2['num_alerts'].fillna(0)
# lack of announces slow zone data = no slow zones
final_data_v2['num_slow_zones'] = final_data_v2['num_slow_zones'].fillna(0)
final_data_v2['total_track_pct'] = final_data_v2['total_track_pct'].fillna(0)
final_data_v2['total_miles_affected'] = final_data_v2['total_miles_affected'].fillna(0)
final_data_v2.head(5)

,service_date,line,est_ridership,otp_numerator,otp_denominator,num_slow_zones,total_track_pct,total_miles_affected,day_of_week,month,otp_score,num_alerts
0,2023-01-01,blue,15214.400000,33536.796260,33747.30964,0.0,0.000000,0.000000,Sunday,January,0.993762,0.0
1,2023-01-01,green,22745.689929,46343.320825,58270.10887,17.0,0.032971,1.780114,Sunday,January,0.795319,0.0
2,2023-01-01,orange,24492.170000,45674.325060,46679.88702,12.0,0.088899,2.003788,Sunday,January,0.978458,2.0
3,2023-01-01,red,27863.440000,59984.367440,66606.05448,23.0,0.074641,3.538731,Sunday,January,0.900584,0.0
4,2023-01-02,blue,19025.400000,67706.968410,68626.25452,0.0,0.000000,0.000000,Monday,January,0.986604,0.0


In [18]:
# ALC
final_data_v2 = pd.merge(
    final_data_v2,
    alerts_cause_effect,
    on=["service_date", "line"],
    how="left"
)

In [19]:
# JJ
# save to CSV
final_data_v2.to_csv("../data/final_merged_data_v2.csv", index=False)

In [20]:
# ALC
# TODO: identify source of NaNs
final_data_v2.corr(numeric_only = True)

,est_ridership,otp_numerator,otp_denominator,num_slow_zones,total_track_pct,total_miles_affected,otp_score,num_alerts,alert_cause_ACCIDENT,alert_cause_CONSTRUCTION,...,alert_effect_PARKING_CLOSURE,alert_effect_PARKING_ISSUE,alert_effect_SCHEDULE_CHANGE,alert_effect_SERVICE_CHANGE,alert_effect_SHUTTLE,alert_effect_STATION_CLOSURE,alert_effect_STATION_ISSUE,alert_effect_STOP_CLOSURE,alert_effect_STOP_MOVE,alert_effect_SUSPENSION
est_ridership,1.000000,0.882259,0.879146,0.421414,-0.097568,0.274147,0.113591,-0.037152,-0.060382,-0.059544,...,-0.174299,-0.023975,NaN,-0.179969,-0.099218,-0.040023,0.108300,0.033143,-0.054044,-0.241715
otp_numerator,0.882259,1.000000,0.991069,0.408791,-0.096116,0.278422,0.159983,-0.038128,-0.057470,-0.083899,...,-0.153443,-0.021292,NaN,-0.207749,-0.085935,-0.082108,0.103974,0.015577,-0.035237,-0.213395
otp_denominator,0.879146,0.991069,1.000000,0.442134,-0.114473,0.316778,0.043726,-0.004952,-0.034617,-0.061518,...,-0.165524,-0.022135,NaN,-0.218992,-0.081994,-0.080093,0.101968,0.017364,-0.040012,-0.193679
num_slow_zones,0.421414,0.408791,0.442134,1.000000,0.137445,0.923036,-0.245081,0.155459,0.041964,0.233504,...,-0.184983,-0.024193,NaN,-0.204792,0.148566,-0.020852,0.198953,0.030221,0.063197,0.021748
total_track_pct,-0.097568,-0.096116,-0.114473,0.137445,1.000000,0.397312,0.159315,-0.143570,-0.033695,0.036772,...,0.038239,0.052520,NaN,0.147084,-0.045734,-0.095331,0.084787,0.014115,0.002608,-0.166206
total_miles_affected,0.274147,0.278422,0.316778,0.923036,0.397312,1.000000,-0.299245,0.161554,0.078273,0.250563,...,-0.148574,-0.010839,NaN,-0.138081,0.121931,-0.019983,0.193736,0.028242,0.092652,0.033296
otp_score,0.113591,0.159983,0.043726,-0.245081,0.159315,-0.299245,1.000000,-0.303250,-0.182760,-0.260387,...,0.120212,0.001275,NaN,0.100987,-0.079669,-0.096281,0.022851,-0.013366,-0.018326,-0.247767
num_alerts,-0.037152,-0.038128,-0.004952,0.155459,-0.143570,0.161554,-0.303250,1.000000,0.179066,0.187267,...,0.089210,0.010354,NaN,0.220144,0.603503,0.083554,0.201156,0.042756,0.103775,0.660842
alert_cause_ACCIDENT,-0.060382,-0.057470,-0.034617,0.041964,-0.033695,0.078273,-0.182760,0.179066,1.000000,0.052601,...,-0.024065,-0.004137,NaN,0.040841,-0.013983,0.075617,0.015407,-0.008283,-0.017157,0.119074
alert_cause_CONSTRUCTION,-0.059544,-0.083899,-0.061518,0.233504,0.036772,0.250563,-0.260387,0.187267,0.052601,1.000000,...,-0.050550,-0.008690,NaN,-0.030082,0.169520,0.150890,0.431835,0.054673,-0.036040,0.205731
